# IBM Before the Breach — time-aware risk ranking

**Goal:** select 500 CVEs from `scoreboard_2026.csv` to maximize later KEV additions. Model only features demonstrably provided at the snapshot date in the *clean baseline*: EPSS, exploit flags, dates derived from `published` and `snapshot_month`. Vulnerability descriptions are **optional / leakage-uncertain**, because today’s NIST texts may have been revised after historical snapshots. The `known_exploited.csv` file is used only for historical labels, not model inputs.

**Labels:** CVEs first entering KEV in the **90 days after the month-end snapshot**. This is a surrogate for the competition’s variable follow-up window and doesn't exactly reproduce the judge's undisclosed 2026 answers. KEV history supplied ends December 2025, so validation snapshots must be through September 2025. The source includes KEV additions occurring before published: a historical candidate is excluded if already in KEV by its snapshot date.

**Run from the directory containing `data/`.** Start with the clean baseline before running experimental text sections. No external data required except downloading the optional Granite model.

In [4]:
# %pip install -U pandas numpy lightgbm scikit-learn matplotlib

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb

DATA = Path("data") if Path("data").exists() else Path("../data")
assert (DATA / "scoreboard_2026.csv").exists(), "Run notebook from project root or change DATA"
SEED=42
HORIZON_DAYS=90
FLAGS=["exploitdb", "metasploit", "nuclei", "poc_github"]


## 1. Build historical snapshots
Each historical candidate is a CVE **published in 2024 or 2025**, observed at the end of its **publication month**. Missing monthly exploit rows are filled with score missing + flags zero, matching scoreboard semantics. The 2025 February source has known missing coverage: keep a `epss_missing` indicator and interpret performance cautiously.


In [5]:
vuln = pd.read_csv(DATA / "vulnerabilities.csv", usecols=["cve_id", "published", "description"], dtype={"cve_id":str, "description":str})
vuln["published"] = pd.to_datetime(vuln["published"], errors="coerce")
vuln = vuln.dropna(subset=["published"]).drop_duplicates("cve_id")

kev = pd.read_csv(DATA / "known_exploited.csv", usecols=["cve_id", "date_added"])
kev["date_added"] = pd.to_datetime(kev["date_added"], errors="coerce")
kev = kev.dropna(subset=["date_added"]).groupby("cve_id", as_index=False)["date_added"].min()

signals = pd.concat([
    pd.read_csv(DATA / f"exploit_signals_{year}.csv", usecols=["month", "cve_id", "epss", *FLAGS])
    for year in (2024, 2025)
], ignore_index=True)
# Avoid accidental duplicate candidate rows, in case source feeds contain duplicates.
signals = signals.drop_duplicates(["month", "cve_id"], keep="last")

hist = vuln[vuln["published"].dt.year.isin([2024, 2025])].copy()
hist["snapshot_month"] = hist["published"].dt.strftime("%Y-%m")
hist = hist.merge(signals, left_on=["cve_id", "snapshot_month"], right_on=["cve_id", "month"], how="left").drop(columns="month")
hist = hist.merge(kev, on="cve_id", how="left")
hist["snapshot_date"] = pd.to_datetime(hist["snapshot_month"] + "-01") + pd.offsets.MonthEnd(0)

# Same eligibility rule as the 2026 scoreboard: NOT in KEV by snapshot date.
hist = hist[hist["date_added"].isna() | (hist["date_added"] > hist["snapshot_date"])].copy()
# Need full 90-day observation period in available 2025 KEV history.
hist = hist[hist["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS) <= pd.Timestamp("2025-12-31")].copy()
hist["label"] = (
    hist["date_added"].notna() &
    (hist["date_added"] > hist["snapshot_date"]) &
    (hist["date_added"] <= hist["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS))
).astype(int)
print("Historical candidates:", len(hist), "labels:", int(hist.label.sum()))
summary = (
    hist.groupby(hist["snapshot_month"].str[:4])["label"]
    .agg(["size", "sum", "mean"])
)

print(summary.to_string())

Historical candidates: 75149 labels: 73
                 size  sum      mean
snapshot_month                      
2024            39868   32  0.000803
2025            35281   41  0.001162


## 2. Feature engineering (clean baseline)
Only scoreboard-supplied, snapshot-time features are used. **Do not** use present-day NIST `nvd_status`, `n_affected_cpes`, `n_references`, current CVSS missingness, current affected products, or reference URLs in this baseline. The NIST `published` date is taken as immutable here; if publication dates can be retrospectively corrected this is another data limitation.


In [6]:
def engineer(df):
    z = df.copy()
    z["published"] = pd.to_datetime(z["published"], errors="coerce")
    z["snapshot_date"] = pd.to_datetime(z["snapshot_month"] + "-01") + pd.offsets.MonthEnd(0)
    z["epss_missing"] = z["epss"].isna().astype("int8")
    z["epss"] = pd.to_numeric(z["epss"], errors="coerce").fillna(0).clip(0, 1)
    z["log_epss"] = np.log10(z["epss"] + 1e-6)
    for f in FLAGS:
        z[f] = pd.to_numeric(z[f], errors="coerce").fillna(0).astype("int8")
    z["any_exploit"] = z[FLAGS].max(axis=1)
    z["n_exploit_sources"] = z[FLAGS].sum(axis=1)
    z["epss_x_exploit"] = z["epss"] * z["any_exploit"]
    z["epss_x_metasploit"] = z["epss"] * z["metasploit"]
    z["age_days"] = (z["snapshot_date"] - z["published"]).dt.days.clip(lower=0)
    z["published_day"] = z["published"].dt.day
    z["published_weekday"] = z["published"].dt.weekday
    z["published_month"] = z["published"].dt.month
    # No snapshot year input: avoid learning annual shifts as a shortcut.
    return z

BASE_FEATURES = ["epss", "log_epss", "epss_missing", *FLAGS,
                 "any_exploit", "n_exploit_sources", "epss_x_exploit",
                 "epss_x_metasploit", "age_days", "published_day",
                 "published_weekday", "published_month"]
hist = engineer(hist)
scoreboard = pd.read_csv(DATA / "scoreboard_2026.csv")
scoreboard = engineer(scoreboard)
assert scoreboard.cve_id.is_unique
print("Historic sample", hist.shape, "scoreboard", scoreboard.shape)


Historic sample (75149, 22) scoreboard (72670, 19)


## 3. Time-based holdout and top-budget metric
Train 2024–June 2025, validate July–September 2025. Holdout top-k scales the competition's 500 / 72,670 selection rate. Also show actual hits and baseline. A 90-day KEV label is **not** the same as the competition's varying-length 2026 follow-up.


In [7]:
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("LightGBM:", lgb.__version__)

model = lgb.LGBMClassifier(n_estimators=10)
print("SUCCESS: LightGBM works!")

NumPy: 2.2.6
Pandas: 2.3.3
Scikit-learn: 1.7.2
LightGBM: 4.7.0
SUCCESS: LightGBM works!


In [8]:

train = hist[hist.snapshot_month <= "2025-06"].copy()
valid = hist[(hist.snapshot_month >= "2025-07") & (hist.snapshot_month <= "2025-09")].copy()
assert len(train) and len(valid) and train.label.sum() and valid.label.sum(), "Not enough positives; inspect label construction."
FRACTION = 500 / 72670

def ranking_report(frame, scores, name, fraction=FRACTION):
    scores = np.asarray(scores, dtype=float)
    k = max(1, int(round(len(frame) * fraction)))
    order = np.argsort(-scores, kind="stable")
    y = frame["label"].to_numpy()
    hits = int(y[order[:k]].sum())
    total = int(y.sum())
    return {"model":name, "n":len(frame), "positives":total, "k":k,
            "hits_at_k":hits, "precision_at_k":hits/k,
            "capture_rate_at_k": hits/total if total else np.nan,
            "average_precision":average_precision_score(y,scores),
            "roc_auc":roc_auc_score(y,scores)}

results=[]
results.append(ranking_report(valid, valid.epss, "EPSS"))
results.append(ranking_report(valid, valid.n_exploit_sources*10 + valid.epss, "Exploit flags -> EPSS"))

params=dict(n_estimators=500, learning_rate=0.025, num_leaves=15,
            max_depth=5, min_child_samples=100, colsample_bytree=0.9,
            subsample=0.9, subsample_freq=1, reg_lambda=5.0,
            random_state=SEED, verbosity=-1, n_jobs=-1)
model=lgb.LGBMClassifier(**params)
model.fit(train[BASE_FEATURES], train.label,
          eval_set=[(valid[BASE_FEATURES],valid.label)],
          eval_metric="average_precision",
          callbacks=[lgb.early_stopping(40, first_metric_only=False, verbose=False)])
valid_pred=model.predict_proba(valid[BASE_FEATURES])[:,1]
results.append(ranking_report(valid, valid_pred, "LightGBM clean"))
display(pd.DataFrame(results).sort_values("hits_at_k",ascending=False))
print("Model best iteration:", model.best_iteration_)


c:\Users\shany\Documents\PatchRader\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
1,Exploit flags -> EPSS,11702,13,81,1,0.012346,0.076923,0.010742,0.834868
2,LightGBM clean,11702,13,81,1,0.012346,0.076923,0.009721,0.749949
0,EPSS,11702,13,81,0,0.000000,0.000000,0.003926,0.700481


Model best iteration: 48


## 4. Feature importance and inspect top candidates
A feature having high importance is not proof of causal value. Compare held-out top-k gains using ablation (remove one group and retrain).

In [12]:
imp=pd.Series(model.feature_importances_, index=BASE_FEATURES).sort_values(ascending=False)
display(imp.to_frame("importance"))
display(valid.assign(predicted_risk=valid_pred)
        .sort_values("predicted_risk",ascending=False)
        [["cve_id","snapshot_month","epss",*FLAGS,"predicted_risk","label"]].head(20))


,importance
published_month,174
epss,129
published_weekday,103
published_day,89
age_days,80
epss_x_exploit,44
log_epss,32
poc_github,5
any_exploit,4
exploitdb,0


,cve_id,snapshot_month,epss,exploitdb,metasploit,nuclei,poc_github,predicted_risk,label
67103,CVE-2025-25231,2025-08,0.08730,0,0,1,1,0.014282,0
67121,CVE-2025-25256,2025-08,0.18910,0,0,1,1,0.014282,0
81261,CVE-2025-54249,2025-09,0.08383,0,0,1,0,0.014282,0
33025,CVE-2024-32640,2025-08,0.92078,0,0,1,1,0.014282,0
81164,CVE-2025-54123,2025-09,0.15579,0,0,1,0,0.014282,0
80936,CVE-2025-53833,2025-07,0.16492,0,0,1,1,0.014282,0
55993,CVE-2025-6970,2025-07,0.16552,0,0,1,1,0.014282,0
80771,CVE-2025-53624,2025-07,0.06330,0,0,1,0,0.014282,0
57004,CVE-2025-8085,2025-09,0.06783,0,0,1,0,0.014282,0
72203,CVE-2025-34077,2025-07,0.16808,1,0,1,1,0.014282,0


## 5. Produce scoreboard top 500 from clean features
Train final model through September 2025 (the last month with complete 90-day labels in supplied KEV history). Fit the number of trees selected on the validation run; no 2026 labels are used.

In [10]:
final_hist=hist[hist.snapshot_month <= "2025-09"].copy()
final_params={**params,"n_estimators": int(model.best_iteration_ or 200)}
final_model=lgb.LGBMClassifier(**final_params)
final_model.fit(final_hist[BASE_FEATURES],final_hist.label)
scoreboard["risk_score_clean"] = final_model.predict_proba(scoreboard[BASE_FEATURES])[:,1]
submission = scoreboard.nlargest(500,"risk_score_clean")[["cve_id"]]
assert len(submission)==500 and submission.cve_id.is_unique
submission.to_csv("top500.csv",index=False)
print("Saved top500.csv")
display(scoreboard.nlargest(15,"risk_score_clean")[["cve_id","risk_score_clean","epss",*FLAGS]])


Saved top500.csv


,cve_id,risk_score_clean,epss,exploitdb,metasploit,nuclei,poc_github
28362,CVE-2026-33439,0.016487,0.07004,0,0,1,1
28877,CVE-2026-3396,0.016487,0.16994,0,0,1,0
31381,CVE-2026-39364,0.016487,0.03614,0,0,1,0
31762,CVE-2026-39808,0.016487,0.13099,0,0,1,1
33684,CVE-2026-42031,0.016487,0.13784,0,0,1,0
35083,CVE-2026-43500,0.016487,0.40266,1,1,0,1
35978,CVE-2026-44578,0.016487,0.05810,0,0,1,1
37681,CVE-2026-4631,0.016487,0.03604,0,0,1,1
42758,CVE-2026-53413,0.016487,0.05604,0,0,0,1
52824,CVE-2026-66804,0.016487,0.05309,0,0,0,1


## 6. OPTIONAL experimental IBM Granite embeddings
**What:** Convert each vulnerability description into 384 semantic features with an IBM Granite encoder. Then reduce to 24 features using PCA fitted on TRAIN ONLY. Test if they improve held-out hits. **WARNING:** `vulnerabilities.csv` contains today's descriptions, not guaranteed point-in-time text. They may include details or confirmed exploitation statements added later. These results are **not leakage-safe**, even if retrospective text is filtered. For a fair production claim obtain archived first-published descriptions / dated vendor advisories, or label text results exploratory only.

Run this only after the clean model. Model weights download on first run. This tests the IBM technology requirement through the real IBM encoder rather than just mentioning Granite.


In [14]:
%pip install -U sentence-transformers scikit-learn
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import re

encoder=SentenceTransformer("ibm-granite/granite-embedding-30m-english")
# Conservative removal of explicit hindsight claims, but this CANNOT guarantee no leakage.
HINDSIGHT = re.compile(r"\b(?:exploited in the wild|actively exploited|known exploited|added to (?:the )?kev|cisa kev|observed in attacks)\b", re.I)
def clean_text(s):
    return HINDSIGHT.sub(" ", str(s) if pd.notna(s) else "")[:1800]

# Encode each CVE once, then join embeddings back by CVE ID.
all_ids=pd.concat([hist[["cve_id","description"]],
                   scoreboard[["cve_id"]].merge(vuln[["cve_id","description"]], on="cve_id",how="left")],
                  ignore_index=True).drop_duplicates("cve_id")
texts=all_ids.description.map(clean_text).tolist()
vec=encoder.encode(texts,batch_size=64,show_progress_bar=True,
                   normalize_embeddings=True,convert_to_numpy=True).astype("float32")
print("Granite embedding matrix:",vec.shape)

id_to_ix=pd.Series(np.arange(len(all_ids)), index=all_ids.cve_id)
tr_ix=id_to_ix.loc[train.cve_id].to_numpy()
va_ix=id_to_ix.loc[valid.cve_id].to_numpy()
sc_ix=id_to_ix.loc[scoreboard.cve_id].to_numpy()

# PCA learned ONLY from training descriptions (no test distribution peeking).
pca=PCA(n_components=24,svd_solver="randomized",random_state=SEED)
pca.fit(vec[tr_ix])
EMB_FEATURES=[f"granite_{i:02d}" for i in range(24)]
def with_embeddings(frame, indices):
    a=frame[BASE_FEATURES].reset_index(drop=True).copy()
    e=pca.transform(vec[indices]).astype("float32")
    a[EMB_FEATURES]=e
    return a

Xtr=with_embeddings(train,tr_ix)
Xva=with_embeddings(valid,va_ix)
text_model=lgb.LGBMClassifier(**params)
text_model.fit(Xtr,train.label,eval_set=[(Xva,valid.label)],
               eval_metric="average_precision",
               callbacks=[lgb.early_stopping(40,verbose=False)])
text_pred=text_model.predict_proba(Xva)[:,1]
display(pd.DataFrame([ranking_report(valid,valid_pred,"Clean baseline"),
                      ranking_report(valid,text_pred,"+ Granite (exploratory)")]))



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
     ------------------- ----------------- 399.4/760.7 kB 12.6 MB/s eta 0:00:01
     --------------------- ---------------- 430.1/760.7 kB 6.8 MB/s eta 0:00:01
     -------------------------- ----------- 532.5/760.7 kB 4.2 MB/s eta 0:00:01
     ------------------------------ ------- 614.4/760.7 kB 3.2 MB/s eta 0:00:01
     --------------------------------- ---- 665.6/760.7 kB 3.0 MB/s eta 0:00:01
     ------------------------------------ - 737.3/760.7 kB 2.7 MB/s eta 0:00:01
     -------------------------------------- 760.7/760.7 kB 2.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
     ---------------------------------------- 0.2/124.1 MB 3.5 MB/s eta 0:00:36
     ---------------------------------------- 0.2/124.1 MB 2.9 MB/s eta 0:00:44
     ---------------------------------------- 0.3/124.1 MB 2.0 MB/s eta 0:01:01
     ---------------------------------------- 0.4/124.

c:\Users\shany\Documents\PatchRader\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\shany\Documents\PatchRader\venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shany\.cache\huggingface\hub\models--ibm-granite--granite-embedding-30m-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administra

Granite embedding matrix: (147819, 384)


c:\Users\shany\Documents\PatchRader\venv\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
0,Clean baseline,11702,13,81,1,0.012346,0.076923,0.009721,0.749949
1,+ Granite (exploratory),11702,13,81,2,0.024691,0.153846,0.013898,0.724027


## 7. OPTIONAL: save experimental Granite submission ONLY if validation improves
If a Granite-enhanced model wins on a holdout with genuinely point-in-time descriptions, retrain PCA on all training data through September 2025 and retrain the model. The below cells produce a separate file for comparison, **not** a validated leakage-safe submission.

In [ ]:
if ranking_report(valid,text_pred,"Granite")["hits_at_k"] > ranking_report(valid,valid_pred,"Clean")["hits_at_k"]:
    final_ix=id_to_ix.loc[final_hist.cve_id].to_numpy()
    pca_final=PCA(n_components=24,svd_solver="randomized",random_state=SEED)
    pca_final.fit(vec[final_ix])
    def make_final_matrix(frame, ix):
        out=frame[BASE_FEATURES].reset_index(drop=True).copy()
        out[EMB_FEATURES]=pca_final.transform(vec[ix]).astype("float32")
        return out
    final_text_model=lgb.LGBMClassifier(**{**params,"n_estimators":int(text_model.best_iteration_ or 200)})
    final_text_model.fit(make_final_matrix(final_hist,final_ix),final_hist.label)
    score_granite=final_text_model.predict_proba(make_final_matrix(scoreboard,sc_ix))[:,1]
    pd.DataFrame({"cve_id":scoreboard.cve_id,"score":score_granite}).nlargest(500,"score")[["cve_id"]].to_csv("top500_granite_experimental.csv",index=False)
    print("Saved top500_granite_experimental.csv (LEAKAGE-UNCERTAIN)")
else:
    print("Granite did not increase held-out hits at the contest budget. Keep the clean submission.")


## Recommended next experiments
1. **Ablate** EPS​S, flags, interactions, text individually and compare hits at same fraction. Holdout positives are rare, so differences of one case are noisy.
2. **Robustness:** repeat on several time folds with equal 90-day label windows, and report aggregated hits, precision, AP, not just one fold.
3. **Vendor momentum:** use only if a reliable *point-in-time* vendor source exists for all CVEs. The present-day `affected_products.csv` is NOT such a source. `known_exploited.csv` vendor is present only for labeled positives; never join it as a feature.
4. **Calibration:** the classifier score is a ranking score, not necessarily calibrated real-world risk probability.
5. **Evaluation caveat:** CISA KEV misses many real exploitations. Treat non-KEV as **not observed positive**, not proven unexploited.
6. **Time shift:** model EPSS changed March 2025; assess post-change-only validation, where enough positive cases exist.
7. **Submission:** inspect `top500.csv` for 500 unique IDs all in scoreboard. Run a demo with evidence-based explanations, not unsupported 'confirmed exploitation' claims.


In [15]:

import numpy as np
import pandas as pd

# Existing validation predictions
# valid_pred = clean LightGBM predictions
# text_pred = LightGBM + Granite predictions

def percentile_rank(pred):
    return pd.Series(pred).rank(pct=True).to_numpy()

clean_r = percentile_rank(valid_pred)
granite_r = percentile_rank(text_pred)

results = []

for granite_weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    combined = (
        (1 - granite_weight) * clean_r
        + granite_weight * granite_r
    )

    results.append(
        ranking_report(
            valid,
            combined,
            f"Granite weight={granite_weight:.2f}"
        )
    )

display(
    pd.DataFrame(results)
    .sort_values(
        ["hits_at_k", "average_precision"],
        ascending=False
    )
)


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
4,Granite weight=1.00,11702,13,81,2,0.024691,0.153846,0.013898,0.724027
3,Granite weight=0.75,11702,13,81,2,0.024691,0.153846,0.013841,0.750469
2,Granite weight=0.50,11702,13,81,2,0.024691,0.153846,0.012815,0.766918
1,Granite weight=0.25,11702,13,81,2,0.024691,0.153846,0.012075,0.763252
0,Granite weight=0.00,11702,13,81,1,0.012346,0.076923,0.009721,0.749949


In [16]:

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Existing Granite embeddings
X_train_emb = vec[tr_ix]
X_valid_emb = vec[va_ix]

scaler = StandardScaler()
X_train_emb = scaler.fit_transform(X_train_emb)
X_valid_emb = scaler.transform(X_valid_emb)

y_train = train["label"].to_numpy().astype("float32")
y_valid = valid["label"].to_numpy().astype("float32")

class GraniteRiskHead(nn.Module):
    def __init__(self, input_dim=384):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = GraniteRiskHead().to(device)

# Cap positive weighting to avoid extreme gradients
n_pos = max(1, y_train.sum())
n_neg = len(y_train) - n_pos

pos_weight = min(n_neg / n_pos, 100.0)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=device)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2
)

dataset = TensorDataset(
    torch.tensor(X_train_emb, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True
)

# Fixed experiment; do not repeatedly tune against
# the same small validation set.
for epoch in range(10):
    model.train()

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

model.eval()

with torch.no_grad():
    Xv = torch.tensor(
        X_valid_emb,
        dtype=torch.float32,
        device=device
    )

    head_pred = torch.sigmoid(model(Xv)).cpu().numpy()

print(
    "Granite head AP:",
    average_precision_score(y_valid, head_pred)
)

display(pd.DataFrame([
    ranking_report(valid, valid_pred, "Clean LightGBM"),
    ranking_report(valid, text_pred, "Granite + LightGBM"),
    ranking_report(valid, head_pred, "Granite neural head")
]))


Granite head AP: 0.015577126586391245


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
0,Clean LightGBM,11702,13,81,1,0.012346,0.076923,0.009721,0.749949
1,Granite + LightGBM,11702,13,81,2,0.024691,0.153846,0.013898,0.724027
2,Granite neural head,11702,13,81,2,0.024691,0.153846,0.015577,0.832324


In [19]:

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
EPOCHS = 12
BATCH_SIZE = 512
LR = 5e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# 2. PREPARE GRANITE EMBEDDINGS
# ============================================================

X_emb_train = vec[tr_ix].astype("float32")
X_emb_valid = vec[va_ix].astype("float32")

emb_scaler = StandardScaler()
X_emb_train = emb_scaler.fit_transform(X_emb_train)
X_emb_valid = emb_scaler.transform(X_emb_valid)

# ============================================================
# 3. PREPARE STRUCTURED FEATURES
# ============================================================

# Use only the snapshot-safe structured features.
# BASE_FEATURES should already be defined in your notebook.

FEATURES = list(BASE_FEATURES)

X_struct_train = train[FEATURES].copy()
X_struct_valid = valid[FEATURES].copy()

# Prevent infinities from breaking preprocessing
X_struct_train = X_struct_train.replace(
    [np.inf, -np.inf], np.nan
)
X_struct_valid = X_struct_valid.replace(
    [np.inf, -np.inf], np.nan
)

# Fit preprocessing exclusively on training data
imputer = SimpleImputer(strategy="constant", fill_value=0)

X_struct_train = imputer.fit_transform(X_struct_train)
X_struct_valid = imputer.transform(X_struct_valid)

struct_scaler = StandardScaler()
X_struct_train = struct_scaler.fit_transform(X_struct_train)
X_struct_valid = struct_scaler.transform(X_struct_valid)

# Optional numerical clipping for stability
X_struct_train = np.clip(X_struct_train, -10, 10).astype("float32")
X_struct_valid = np.clip(X_struct_valid, -10, 10).astype("float32")

y_train = train["label"].to_numpy(dtype="float32")
y_valid = valid["label"].to_numpy(dtype="float32")

print("Training samples:", len(y_train))
print("Training positives:", int(y_train.sum()))
print("Granite dimensions:", X_emb_train.shape[1])
print("Structured dimensions:", X_struct_train.shape[1])

# ============================================================
# 4. DEFINE TWO-BRANCH FUSION NETWORK
# ============================================================

class GraniteFusionNet(nn.Module):
    def __init__(self, emb_dim=384, struct_dim=15):
        super().__init__()

        # Semantic branch
        self.text_branch = nn.Sequential(
            nn.Linear(emb_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Structured feature branch
        self.structured_branch = nn.Sequential(
            nn.Linear(struct_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Fusion branch
        self.fusion = nn.Sequential(
            nn.Linear(32 + 16, 32),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(32, 1)
        )

    def forward(self, text, structured):
        text_features = self.text_branch(text)
        structured_features = self.structured_branch(structured)

        combined = torch.cat(
            [text_features, structured_features],
            dim=1
        )

        return self.fusion(combined).squeeze(-1)


fusion_model = GraniteFusionNet(
    emb_dim=X_emb_train.shape[1],
    struct_dim=X_struct_train.shape[1]
).to(device)

print(fusion_model)

# ============================================================
# 5. CLASS IMBALANCE HANDLING
# ============================================================

n_pos = max(1, y_train.sum())
n_neg = len(y_train) - n_pos

# Avoid an extremely large positive-class weight
pos_weight = min(n_neg / n_pos, 100.0)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        pos_weight,
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.AdamW(
    fusion_model.parameters(),
    lr=LR,
    weight_decay=0.02
)

# ============================================================
# 6. CREATE DATALOADERS
# ============================================================

train_dataset = TensorDataset(
    torch.tensor(X_emb_train, dtype=torch.float32),
    torch.tensor(X_struct_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_text_tensor = torch.tensor(
    X_emb_valid, dtype=torch.float32, device=device
)

valid_struct_tensor = torch.tensor(
    X_struct_valid, dtype=torch.float32, device=device
)

# ============================================================
# 7. TRAIN
# ============================================================

history = []

for epoch in range(EPOCHS):
    fusion_model.train()
    running_loss = 0.0

    for text_batch, struct_batch, target_batch in train_loader:
        text_batch = text_batch.to(device)
        struct_batch = struct_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()

        logits = fusion_model(text_batch, struct_batch)
        loss = criterion(logits, target_batch)

        loss.backward()
        nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item() * len(target_batch)

    fusion_model.eval()

    with torch.no_grad():
        logits = fusion_model(
            valid_text_tensor,
            valid_struct_tensor
        )
        predictions = torch.sigmoid(logits).cpu().numpy()

    ap = average_precision_score(y_valid, predictions)
    auc = roc_auc_score(y_valid, predictions)

    history.append({
        "epoch": epoch + 1,
        "train_loss": running_loss / len(train_dataset),
        "valid_ap": ap,
        "valid_auc": auc
    })

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {history[-1]['train_loss']:.5f} | "
        f"AP: {ap:.6f} | "
        f"AUC: {auc:.6f}"
    )

# ============================================================
# 8. FINAL EVALUATION
# ============================================================

fusion_pred = predictions

comparison = pd.DataFrame([
    ranking_report(
        valid, valid_pred, "Clean LightGBM"
    ),
    ranking_report(
        valid, text_pred, "Granite + LightGBM"
    ),
    ranking_report(
        valid, head_pred, "Granite neural head"
    ),
    ranking_report(
        valid, fusion_pred, "Granite Fusion Network"
    )
])

display(
    comparison.sort_values(
        ["hits_at_k", "average_precision"],
        ascending=False
    )
)

# Training history
display(pd.DataFrame(history))


Device: cpu
Training samples: 63447
Training positives: 60
Granite dimensions: 384
Structured dimensions: 15
GraniteFusionNet(
  (text_branch): Sequential(
    (0): Linear(in_features=384, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.35, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
  )
  (structured_branch): Sequential(
    (0): Linear(in_features=15, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
  )
  (fusion): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.35, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)
Epoch 01 | Loss: 0.46220 | AP: 0.008932 | AUC: 0.774963
Epoch 02 | Loss: 0.26400 | AP: 0.004527 | AUC: 0.805504
Epoch 03 | Loss: 0.19038 | AP: 0.006045 | AUC: 0.832295
Epoch 04 | Loss: 0.14800 | AP: 0.009993 | AU

,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
2,Granite neural head,11702,13,81,2,0.024691,0.153846,0.015577,0.832324
1,Granite + LightGBM,11702,13,81,2,0.024691,0.153846,0.013898,0.724027
0,Clean LightGBM,11702,13,81,1,0.012346,0.076923,0.009721,0.749949
3,Granite Fusion Network,11702,13,81,0,0.000000,0.000000,0.006679,0.832861


,epoch,train_loss,valid_ap,valid_auc
0,1,0.462203,0.008932,0.774963
1,2,0.263997,0.004527,0.805504
2,3,0.190380,0.006045,0.832295
3,4,0.148000,0.009993,0.849556
4,5,0.125578,0.010257,0.858736
5,6,0.078995,0.008881,0.857420
6,7,0.087261,0.007952,0.856992
7,8,0.058127,0.009386,0.838421
8,9,0.031801,0.008199,0.842673
9,10,0.032371,0.006789,0.832262


In [20]:

def rank_normalize(scores):
    return pd.Series(scores).rank(pct=True).to_numpy()

fusion_r = rank_normalize(fusion_pred)
head_r = rank_normalize(head_pred)

blend_results = []

for w in [0, 0.25, 0.5, 0.75, 1]:
    combined = w * fusion_r + (1 - w) * head_r

    blend_results.append(
        ranking_report(
            valid,
            combined,
            f"Fusion weight = {w:.2f}"
        )
    )

display(pd.DataFrame(blend_results))


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
0,Fusion weight = 0.00,11702,13,81,2,0.024691,0.153846,0.015577,0.832324
1,Fusion weight = 0.25,11702,13,81,0,0.000000,0.000000,0.007576,0.835680
2,Fusion weight = 0.50,11702,13,81,0,0.000000,0.000000,0.007329,0.837316
3,Fusion weight = 0.75,11702,13,81,0,0.000000,0.000000,0.007041,0.836059
4,Fusion weight = 1.00,11702,13,81,0,0.000000,0.000000,0.006679,0.832861


In [22]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

# =====================================================
# 1. PREPARE DATA
# =====================================================

SEED = 42

ytr = train["label"].to_numpy().astype(int)
yva = valid["label"].to_numpy().astype(int)

# Full Granite embeddings
emb_train = vec[tr_ix].astype("float32")
emb_valid = vec[va_ix].astype("float32")

emb_scaler = StandardScaler()
emb_train = emb_scaler.fit_transform(emb_train)
emb_valid = emb_scaler.transform(emb_valid)

# Structured features
features = list(BASE_FEATURES)

struct_train = train[features].replace(
    [np.inf, -np.inf], np.nan
)
struct_valid = valid[features].replace(
    [np.inf, -np.inf], np.nan
)

imputer = SimpleImputer(
    strategy="constant",
    fill_value=0
)

struct_train = imputer.fit_transform(struct_train)
struct_valid = imputer.transform(struct_valid)

struct_scaler = StandardScaler()

struct_train = struct_scaler.fit_transform(struct_train)
struct_valid = struct_scaler.transform(struct_valid)

struct_train = np.clip(struct_train, -10, 10)
struct_valid = np.clip(struct_valid, -10, 10)

# Limit positive weights to avoid extreme fitting
n_pos = ytr.sum()
n_neg = len(ytr) - n_pos

positive_weight = min(n_neg / n_pos, 100.0)
sample_weights = np.where(ytr == 1, positive_weight, 1.0)

print("Training positives:", n_pos)
print("Positive weight:", positive_weight)

# =====================================================
# 2. MODEL A — GRANITE + LOGISTIC REGRESSION
# =====================================================

model_a = LogisticRegression(
    C=0.03,
    max_iter=2000,
    solver="lbfgs",
    random_state=SEED
)

model_a.fit(
    emb_train,
    ytr,
    sample_weight=sample_weights
)

pred_a = model_a.predict_proba(emb_valid)[:, 1]

# =====================================================
# 3. MODEL B — GRANITE + STRUCTURED LOGISTIC REGRESSION
# =====================================================

# Scale the structured branch contribution initially
# so the extra 15 features don't dominate the embeddings.

STRUCT_WEIGHT = 0.5

Xtr_combined = np.hstack([
    emb_train,
    struct_train * STRUCT_WEIGHT
])

Xva_combined = np.hstack([
    emb_valid,
    struct_valid * STRUCT_WEIGHT
])

model_b = LogisticRegression(
    C=0.03,
    max_iter=2000,
    solver="lbfgs",
    random_state=SEED
)

model_b.fit(
    Xtr_combined,
    ytr,
    sample_weight=sample_weights
)

pred_b = model_b.predict_proba(Xva_combined)[:, 1]

# =====================================================
# 4. MODEL C — GRANITE + XGBOOST
# =====================================================

# Reduce embeddings to avoid feeding 384 features
# into trees trained on only ~60 positive examples.

pca = PCA(
    n_components=16,
    random_state=SEED
)

emb_train_pca = pca.fit_transform(emb_train)
emb_valid_pca = pca.transform(emb_valid)

Xtr_xgb = np.hstack([
    emb_train_pca,
    struct_train
])

Xva_xgb = np.hstack([
    emb_valid_pca,
    struct_valid
])

model_c = XGBClassifier(
    n_estimators=150,
    max_depth=2,
    learning_rate=0.03,
    min_child_weight=10,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=20,
    reg_alpha=2,
    scale_pos_weight=positive_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

model_c.fit(Xtr_xgb, ytr)

pred_c = model_c.predict_proba(Xva_xgb)[:, 1]

# =====================================================
# 5. COMPARE ALL MODELS
# =====================================================

results = pd.DataFrame([
    ranking_report(
        valid, head_pred, "Granite neural head"
    ),
    ranking_report(
        valid, pred_a, "Granite + Logistic"
    ),
    ranking_report(
        valid, pred_b, "Granite + Structured Logistic"
    ),
    ranking_report(
        valid, pred_c, "Granite + XGBoost"
    )
])

display(
    results.sort_values(
        ["hits_at_k", "average_precision"],
        ascending=False
    )
)

# =====================================================
# 6. RANK THE POSITIVE CVEs
# =====================================================

positive_ranks = valid[["cve_id", "label"]].reset_index(drop=True).copy()

predictions = {
    "Neural Head": head_pred,
    "Granite Logistic": pred_a,
    "Structured Logistic": pred_b,
    "XGBoost": pred_c
}

for name, scores in predictions.items():
    positive_ranks[name] = (
        pd.Series(scores)
        .rank(ascending=False, method="min")
        .astype(int)
        .to_numpy()
    )

positive_ranks = positive_ranks[
    positive_ranks["label"] == 1
].sort_values("Neural Head")

display(positive_ranks)


Training positives: 60
Positive weight: 100.0


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
0,Granite neural head,11702,13,81,2,0.024691,0.153846,0.015577,0.832324
3,Granite + XGBoost,11702,13,81,2,0.024691,0.153846,0.010172,0.772087
2,Granite + Structured Logistic,11702,13,81,1,0.012346,0.076923,0.005604,0.823338
1,Granite + Logistic,11702,13,81,0,0.000000,0.000000,0.004506,0.800917


,cve_id,label,Neural Head,Granite Logistic,Structured Logistic,XGBoost
1540,CVE-2025-6205,1,14,213,945,3932
1539,CVE-2025-6204,1,32,726,2700,2477
7209,CVE-2025-41244,1,427,2854,450,426
3684,CVE-2025-9377,1,473,645,317,260
3604,CVE-2025-9242,1,545,791,1168,3557
9740,CVE-2025-54236,1,724,204,57,91
8101,CVE-2025-48703,1,858,1277,376,73
5053,CVE-2025-21043,1,927,1382,2853,5080
5052,CVE-2025-21042,1,951,1384,2829,5024
10278,CVE-2025-55177,1,3364,5006,6220,2756


In [23]:

import numpy as np
import pandas as pd

scores = {
    "Granite Head": np.asarray(head_pred),
    "XGBoost": np.asarray(pred_c),
    "Structured Logistic": np.asarray(pred_b)
}

def ranks_desc(x):
    return pd.Series(x).rank(
        ascending=False,
        method="average"
    ).to_numpy()

def normalized_rank(x):
    n = len(x)
    return (n - ranks_desc(x)) / max(n - 1, 1)

# Individual model rankings
rank_scores = {
    name: normalized_rank(pred)
    for name, pred in scores.items()
}

# Reciprocal Rank Fusion: higher is better
def reciprocal_rank_fusion(names, k=60):
    return sum(
        1.0 / (k + ranks_desc(scores[name]))
        for name in names
    )

# Candidate ensembles
ensemble_predictions = {
    "Head only": rank_scores["Granite Head"],

    "Head + XGB average":
        0.5 * rank_scores["Granite Head"]
        + 0.5 * rank_scores["XGBoost"],

    "Head + XGB RRF":
        reciprocal_rank_fusion(
            ["Granite Head", "XGBoost"]
        ),

    "Head + XGB maximum":
        np.maximum(
            rank_scores["Granite Head"],
            rank_scores["XGBoost"]
        ),

    "All 3 RRF":
        reciprocal_rank_fusion(
            [
                "Granite Head",
                "XGBoost",
                "Structured Logistic"
            ]
        ),
}

results = pd.DataFrame([
    ranking_report(valid, pred, name)
    for name, pred in ensemble_predictions.items()
])

display(
    results.sort_values(
        ["hits_at_k", "average_precision"],
        ascending=False
    )
)

# Which positives does each ensemble detect?
positive_comparison = (
    valid[["cve_id", "label"]]
    .reset_index(drop=True)
    .copy()
)

for name, pred in ensemble_predictions.items():
    positive_comparison[name] = ranks_desc(pred).astype(int)

display(
    positive_comparison[
        positive_comparison["label"] == 1
    ].sort_values("Head only")
)


,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
3,Head + XGB maximum,11702,13,81,3,0.037037,0.230769,0.016469,0.849263
0,Head only,11702,13,81,2,0.024691,0.153846,0.015577,0.832324
2,Head + XGB RRF,11702,13,81,1,0.012346,0.076923,0.014710,0.854683
4,All 3 RRF,11702,13,81,0,0.000000,0.000000,0.011976,0.861395
1,Head + XGB average,11702,13,81,0,0.000000,0.000000,0.008241,0.843508


,cve_id,label,Head only,Head + XGB average,Head + XGB RRF,Head + XGB maximum,All 3 RRF
1540,CVE-2025-6205,1,14,1210,43,26,106
1539,CVE-2025-6204,1,32,655,88,61,172
7209,CVE-2025-41244,1,427,150,428,756,433
3684,CVE-2025-9377,1,473,127,333,474,308
3604,CVE-2025-9242,1,545,1286,1050,954,1124
9740,CVE-2025-54236,1,724,146,167,176,85
8101,CVE-2025-48703,1,858,167,146,142,184
5053,CVE-2025-21043,1,927,2206,1760,1577,2116
5052,CVE-2025-21042,1,951,2191,1794,1612,2142
10278,CVE-2025-55177,1,3364,2271,3476,4274,4324


In [24]:
# Rolling-origin backtest for the existing IBM Before the Breach notebook.
# Requires: hist, vec, id_to_ix, BASE_FEATURES, and ranking_report (the latter optional).
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

HORIZON_DAYS = 90
BUDGET_SHARE = 500 / 72670
SEED = 42
EPOCHS = 10

# The last fold is the previously inspected cohort, so it is diagnostic, NOT an independent test.
FOLDS = [
    ('2025-Q1', '2025-01-01', '2025-04-01'),
    ('2025-Q2', '2025-04-01', '2025-07-01'),
    ('2025-Q3 (previously inspected)', '2025-07-01', '2025-10-01'),
]

def report(y, scores, name, k):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind='stable')
    hits = int(y[order[:k]].sum())
    return dict(model=name, n=len(y), positives=int(y.sum()), k=k,
                hits_at_k=hits, precision_at_k=hits/k,
                capture_rate_at_k=hits/max(1,int(y.sum())),
                average_precision=average_precision_score(y,scores) if y.sum() else np.nan,
                roc_auc=roc_auc_score(y,scores) if len(np.unique(y))>1 else np.nan)

def rank_pct(x):
    return pd.Series(np.asarray(x)).rank(pct=True, method='average').to_numpy()

def exploit_first_scores(frame):
    # Exploit present first; EPSS breaks ties; deterministic tiny tie breaker omitted.
    flags = frame[['exploitdb','metasploit','nuclei','poc_github']].fillna(0).astype(float)
    any_exploit = flags.gt(0).any(axis=1).to_numpy().astype(int)
    epss = pd.to_numeric(frame['epss'], errors='coerce').fillna(-1).to_numpy()
    # Convert lexicographic priority into ordering-based numerical scores.
    order = np.lexsort((-epss, -any_exploit))
    out = np.empty(len(frame), dtype=float)
    out[order] = np.arange(len(frame), 0, -1)
    return out

class Head(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,32),nn.ReLU(),nn.Dropout(.3),nn.Linear(32,1))
    def forward(self,x):
        return self.net(x).squeeze(-1)

def fit_head(X, y, Xval, seed=42, epochs=10):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = Head(X.shape[1])
    npos = max(1, int(y.sum()))
    weight = min((len(y)-npos)/npos, 100)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(weight)))
    opt = torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-2)
    loader=DataLoader(TensorDataset(torch.from_numpy(X.astype('float32')),
                                    torch.from_numpy(y.astype('float32'))),
                      batch_size=512,shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb,yb in loader:
            opt.zero_grad(); loss=criterion(model(xb),yb); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.from_numpy(Xval.astype('float32')))).numpy()

frame=hist.copy()
frame['snapshot_date']=pd.to_datetime(frame['snapshot_month'].astype(str)+'-01',errors='coerce')
frame=frame.dropna(subset=['snapshot_date','cve_id','label']).copy()
# Use historical records once per CVE. This works for publish-month snapshots.
# If your hist has repeated monthly observations per CVE, DO NOT deduplicate without
# redesigning the cohort and validating grouped, point-in-time features.
if frame.cve_id.duplicated().any():
    raise ValueError('hist has repeated CVEs; this backtest expects one publish-month snapshot per CVE.')

rows=[]; positive_ranks=[]
for fold_name,start_s,end_s in FOLDS:
    start,end=pd.Timestamp(start_s),pd.Timestamp(end_s)
    # Conservative label maturation: train only on observations whose entire
    # 90-day target horizon had finished BEFORE the validation period.
    train_cutoff = start - pd.Timedelta(days=HORIZON_DAYS)
    tr=frame.loc[(frame.snapshot_date + pd.offsets.MonthEnd(0) + pd.Timedelta(days=HORIZON_DAYS)) < start].copy()
    va=frame.loc[(frame.snapshot_date>=start)&(frame.snapshot_date<end)].copy()
    tr=tr.reset_index(drop=True); va=va.reset_index(drop=True)
    if tr.label.sum()<5 or va.label.sum()<1:
        print(f'Skipping {fold_name}: train positives={tr.label.sum()}, validation positives={va.label.sum()}')
        continue
    ytr=tr.label.to_numpy(dtype=int); yva=va.label.to_numpy(dtype=int)
    tr_ix_fold=id_to_ix.loc[tr.cve_id].to_numpy()
    va_ix_fold=id_to_ix.loc[va.cve_id].to_numpy()
    emb_tr=vec[tr_ix_fold].astype('float32')
    emb_va=vec[va_ix_fold].astype('float32')
    scaler=StandardScaler()
    emb_tr=scaler.fit_transform(emb_tr).astype('float32')
    emb_va=scaler.transform(emb_va).astype('float32')
    head_scores=fit_head(emb_tr,ytr,emb_va,seed=SEED,epochs=EPOCHS)
    imputer=SimpleImputer(strategy='constant',fill_value=0)
    str_tr=tr[BASE_FEATURES].replace([np.inf,-np.inf],np.nan)
    str_va=va[BASE_FEATURES].replace([np.inf,-np.inf],np.nan)
    str_tr=imputer.fit_transform(str_tr)
    str_va=imputer.transform(str_va)
    ss=StandardScaler()
    str_tr=np.clip(ss.fit_transform(str_tr),-10,10)
    str_va=np.clip(ss.transform(str_va),-10,10)
    pca=PCA(n_components=16,random_state=SEED)
    xtr=np.column_stack((pca.fit_transform(emb_tr),str_tr))
    xva=np.column_stack((pca.transform(emb_va),str_va))
    pos_weight=min((len(ytr)-ytr.sum())/max(1,ytr.sum()),100)
    xgb=XGBClassifier(n_estimators=150,max_depth=2,learning_rate=.03,
        min_child_weight=10,subsample=.85,colsample_bytree=.85,
        reg_lambda=20,reg_alpha=2,scale_pos_weight=pos_weight,
        objective='binary:logistic',eval_metric='aucpr',random_state=SEED,
        n_jobs=-1,tree_method='hist')
    xgb.fit(xtr,ytr)
    xgb_scores=xgb.predict_proba(xva)[:,1]
    max_scores=np.maximum(rank_pct(head_scores),rank_pct(xgb_scores))
    epss_scores=pd.to_numeric(va.epss,errors='coerce').fillna(-1).to_numpy()
    exploit_scores=exploit_first_scores(va)
    k=max(1,round(len(va)*BUDGET_SHARE))
    print(f'{fold_name}: train={len(tr):,} ({int(ytr.sum())} positives), '
          f'validation={len(va):,} ({int(yva.sum())} positives), budget={k}')
    for name,s in [('EPSS',epss_scores),('Exploit first -> EPSS',exploit_scores),
                   ('Granite head',head_scores),('Granite XGBoost',xgb_scores),
                   ('Head + XGB maximum',max_scores)]:
        rows.append(dict(fold=fold_name,**report(yva,s,name,k)))
        rank=pd.Series(s).rank(ascending=False,method='min').astype(int).to_numpy()
        for cv,yy,rr in zip(va.cve_id,yva,rank):
            if yy: positive_ranks.append(dict(fold=fold_name,model=name,cve_id=cv,rank=int(rr),top_k=int(rr<=k)))

results=pd.DataFrame(rows)
if len(results):
    display(results.sort_values(['fold','hits_at_k','average_precision'],ascending=[True,False,False]))
    summary=(results.groupby('model',as_index=False).agg(
        folds=('fold','nunique'),total_hits=('hits_at_k','sum'),
        total_budget=('k','sum'),total_positives=('positives','sum'),
        mean_average_precision=('average_precision','mean')))
    summary['pooled_precision_at_budget']=summary.total_hits/summary.total_budget
    summary['pooled_capture_rate']=summary.total_hits/summary.total_positives
    display(summary.sort_values('total_hits',ascending=False))
    display(pd.DataFrame(positive_ranks).sort_values(['fold','cve_id','model']))
else:
    print('No folds eligible. Inspect snapshot construction and positive labels.')


2025-Q1: train=28,839 (20 positives), validation=11,913 (14 positives), budget=82
2025-Q2: train=39,868 (32 positives), validation=11,666 (14 positives), budget=80
2025-Q3 (previously inspected): train=51,781 (46 positives), validation=11,702 (13 positives), budget=81


,fold,model,n,positives,k,hits_at_k,precision_at_k,capture_rate_at_k,average_precision,roc_auc
1,2025-Q1,Exploit first -> EPSS,11913,14,82,3,0.036585,0.214286,0.087491,0.694986
0,2025-Q1,EPSS,11913,14,82,2,0.024390,0.142857,0.082303,0.689022
2,2025-Q1,Granite head,11913,14,82,1,0.012195,0.071429,0.005836,0.797216
3,2025-Q1,Granite XGBoost,11913,14,82,0,0.000000,0.000000,0.007414,0.849027
4,2025-Q1,Head + XGB maximum,11913,14,82,0,0.000000,0.000000,0.006255,0.881473
6,2025-Q2,Exploit first -> EPSS,11666,14,80,6,0.075000,0.428571,0.135093,0.785420
5,2025-Q2,EPSS,11666,14,80,6,0.075000,0.428571,0.106550,0.767348
8,2025-Q2,Granite XGBoost,11666,14,80,5,0.062500,0.357143,0.059253,0.805024
9,2025-Q2,Head + XGB maximum,11666,14,80,4,0.050000,0.285714,0.031185,0.826829
7,2025-Q2,Granite head,11666,14,80,1,0.012500,0.071429,0.006816,0.633601


,model,folds,total_hits,total_budget,total_positives,mean_average_precision,pooled_precision_at_budget,pooled_capture_rate
1,Exploit first -> EPSS,3,11,243,41,0.077964,0.045267,0.268293
0,EPSS,3,8,243,41,0.064260,0.032922,0.195122
2,Granite XGBoost,3,6,243,41,0.025294,0.024691,0.146341
4,Head + XGB maximum,3,5,243,41,0.014349,0.020576,0.121951
3,Granite head,3,2,243,41,0.005266,0.008230,0.048780


,fold,model,cve_id,rank,top_k
0,2025-Q1,EPSS,CVE-2024-13159,3455,0
14,2025-Q1,Exploit first -> EPSS,CVE-2024-13159,4011,0
42,2025-Q1,Granite XGBoost,CVE-2024-13159,1130,0
28,2025-Q1,Granite head,CVE-2024-13159,1544,0
56,2025-Q1,Head + XGB maximum,CVE-2024-13159,1969,0
...,...,...,...,...,...
143,2025-Q3 (previously inspected),EPSS,CVE-2025-9377,382,0
156,2025-Q3 (previously inspected),Exploit first -> EPSS,CVE-2025-9377,613,0
182,2025-Q3 (previously inspected),Granite XGBoost,CVE-2025-9377,253,0
169,2025-Q3 (previously inspected),Granite head,CVE-2025-9377,522,0


In [25]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

signals = pd.concat([
    pd.read_csv(DATA_DIR / "exploit_signals_2024.csv"),
    pd.read_csv(DATA_DIR / "exploit_signals_2025.csv")
], ignore_index=True)

vuln_dates = pd.read_csv(
    DATA_DIR / "vulnerabilities.csv",
    usecols=["cve_id", "published"]
)

kev_dates = pd.read_csv(
    DATA_DIR / "known_exploited.csv",
    usecols=["cve_id", "date_added"]
)

vuln_dates["published"] = pd.to_datetime(
    vuln_dates["published"], errors="coerce"
)
kev_dates["date_added"] = pd.to_datetime(
    kev_dates["date_added"], errors="coerce"
)

signals["snapshot_date"] = (
    pd.to_datetime(signals["month"] + "-01")
    + pd.offsets.MonthEnd(0)
)

df = signals.merge(vuln_dates, on="cve_id", how="left")
df = df.merge(kev_dates, on="cve_id", how="left")

# Only CVEs published by the prediction snapshot
df = df[df["published"].le(df["snapshot_date"])].copy()

# CVEs already in KEV are not prediction candidates
df = df[
    df["date_added"].isna()
    | (df["date_added"] > df["snapshot_date"])
].copy()

# Require 90-day follow-up within known label coverage
LABEL_CUTOFF = pd.Timestamp("2025-12-31")
df["horizon_end"] = df["snapshot_date"] + pd.Timedelta(days=90)
df = df[df["horizon_end"] <= LABEL_CUTOFF].copy()

df["future_kev_90d"] = (
    df["date_added"].notna()
    & (df["date_added"] > df["snapshot_date"])
    & (df["date_added"] <= df["horizon_end"])
).astype("int8")

summary = df.groupby("month").agg(
    observations=("cve_id", "size"),
    unique_cves=("cve_id", "nunique"),
    positive_rows=("future_kev_90d", "sum")
)

print("Monthly historical snapshots:")
display(summary)

print("Total eligible observations:", len(df))
print("Unique CVEs:", df["cve_id"].nunique())
print("Positive snapshot rows:", df["future_kev_90d"].sum())
print(
    "Unique future-positive CVEs:",
    df.loc[df["future_kev_90d"] == 1, "cve_id"].nunique()
)


Monthly historical snapshots:


,observations,unique_cves,positive_rows
month,,,
2024-01,222079,222079,10
2024-02,224887,224887,12
2024-03,228174,228174,12
2024-04,231768,231768,13
2024-05,236353,236353,15
2024-06,239388,239388,26
2024-07,242256,242256,28
2024-08,245354,245354,32
2024-09,247809,247809,26


Total eligible observations: 5375979
Unique CVEs: 294596
Positive snapshot rows: 516
Unique future-positive CVEs: 217


In [26]:

import numpy as np
import pandas as pd

audit = df.copy()

audit["age_days"] = (
    audit["snapshot_date"] - audit["published"]
).dt.days

# Group by age at the time of prediction
audit["age_group"] = pd.cut(
    audit["age_days"],
    bins=[-1, 30, 90, 180, 365, 730, np.inf],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "1-2 years",
        "2+ years"
    ]
)

summary = (
    audit.groupby("age_group", observed=True)
    .agg(
        observations=("cve_id", "size"),
        positive_rows=("future_kev_90d", "sum")
    )
)

summary["positive_rate"] = (
    summary["positive_rows"] /
    summary["observations"]
)

summary["positive_unique_cves"] = (
    audit[audit["future_kev_90d"] == 1]
    .groupby("age_group", observed=True)["cve_id"]
    .nunique()
)

display(summary.fillna(0))

# Compare vulnerabilities published in the snapshot year
# against older vulnerabilities
audit["published_year"] = audit["published"].dt.year
audit["snapshot_year"] = audit["snapshot_date"].dt.year

audit["same_year"] = (
    audit["published_year"] == audit["snapshot_year"]
)

display(
    audit.groupby(["snapshot_year", "same_year"])
    .agg(
        rows=("cve_id", "size"),
        positives=("future_kev_90d", "sum"),
        positive_unique_cves=(
            "cve_id",
            lambda ids: ids[
                audit.loc[ids.index, "future_kev_90d"] == 1
            ].nunique()
        )
    )
)


,observations,positive_rows,positive_rate,positive_unique_cves
age_group,,,,
0-30 days,71448,70,0.000980,70
31-90 days,142025,67,0.000472,48
91-180 days,203914,54,0.000265,31
181-365 days,377775,54,0.000143,25
1-2 years,610272,41,0.000067,17
2+ years,3970545,230,0.000058,83


rows  positives  positive_unique_cves
snapshot_year same_year                                          
2024          False      2635027        161                    66
              True        248612        100                    53
2025          False      2319000        172                    73
              True        173340         83                    49

In [27]:

import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("data") if Path("data").exists() else Path("../data")

vuln = pd.read_csv(
    DATA / "vulnerabilities.csv",
    usecols=["cve_id", "published"]
)
kev = pd.read_csv(
    DATA / "known_exploited.csv",
    usecols=["cve_id", "date_added"]
)
signals = pd.concat([
    pd.read_csv(DATA / "exploit_signals_2024.csv"),
    pd.read_csv(DATA / "exploit_signals_2025.csv")
], ignore_index=True)

vuln["published"] = pd.to_datetime(vuln["published"])
kev["date_added"] = pd.to_datetime(kev["date_added"])
kev = kev.drop_duplicates("cve_id")

# Match the benchmark: one snapshot in the CVE's
# publication month.
cohort = vuln[
    vuln["published"].between(
        "2024-01-01", "2025-09-30"
    )
].copy()

cohort["snapshot_month"] = (
    cohort["published"].dt.strftime("%Y-%m")
)

# February 2025 has major missing signal coverage.
# Exclude it from this initial experiment.
cohort = cohort[
    cohort["snapshot_month"] != "2025-02"
]

features_raw = [
    "epss", "exploitdb", "metasploit",
    "nuclei", "poc_github"
]

signals = signals.rename(
    columns={"month": "snapshot_month"}
)

cohort = cohort.merge(
    signals[["snapshot_month", "cve_id"] + features_raw],
    on=["cve_id", "snapshot_month"],
    how="left",
    validate="one_to_one"
)

cohort = cohort.merge(
    kev, on="cve_id", how="left",
    validate="one_to_one"
)

cohort["snapshot_date"] = (
    cohort["published"]
    + pd.offsets.MonthEnd(0)
)

# Remove CVEs already known to be exploited.
cohort = cohort[
    cohort["date_added"].isna()
    | (cohort["date_added"] > cohort["snapshot_date"])
].copy()

# 90-day future KEV label
cohort["label"] = (
    cohort["date_added"].notna()
    & (cohort["date_added"] > cohort["snapshot_date"])
    & (
        cohort["date_added"]
        <= cohort["snapshot_date"] + pd.Timedelta(days=90)
    )
).astype(int)

# Features available in the 2026 scoreboard
cohort["age_days"] = (
    cohort["snapshot_date"] - cohort["published"]
).dt.days

cohort["epss_missing"] = cohort["epss"].isna().astype(int)
cohort["epss"] = cohort["epss"].fillna(0)

flags = ["exploitdb", "metasploit", "nuclei", "poc_github"]
cohort[flags] = cohort[flags].fillna(0)

cohort["n_exploit_sources"] = cohort[flags].sum(axis=1)
cohort["any_exploit"] = (
    cohort["n_exploit_sources"] > 0
).astype(int)

cohort["log_epss"] = np.log1p(cohort["epss"] * 1000)
cohort["epss_x_exploit"] = (
    cohort["epss"] * cohort["any_exploit"]
)

FEATURES = [
    "epss", "log_epss", "epss_missing",
    "exploitdb", "metasploit", "nuclei",
    "poc_github", "n_exploit_sources",
    "any_exploit", "epss_x_exploit",
    "age_days"
]

train_new = cohort[
    cohort["snapshot_month"] < "2025-07"
].copy()

valid_new = cohort[
    cohort["snapshot_month"] >= "2025-07"
].copy()

print("Train:", train_new.shape)
print("Validation:", valid_new.shape)
print("Training positives:", train_new.label.sum())
print("Validation positives:", valid_new.label.sum())


Train: (59783, 17)
Validation: (11702, 17)
Training positives: 57
Validation positives: 13


In [28]:

from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

X_train = train_new[FEATURES]
y_train = train_new["label"]

X_valid = valid_new[FEATURES]
y_valid = valid_new["label"]

n_pos = max(1, y_train.sum())
n_neg = len(y_train) - n_pos

model = XGBClassifier(
    n_estimators=250,
    max_depth=2,
    learning_rate=0.025,
    min_child_weight=15,
    subsample=0.85,
    colsample_bytree=0.9,
    reg_lambda=30,
    reg_alpha=5,
    scale_pos_weight=min(n_neg / n_pos, 100),
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
xgb_pred = model.predict_proba(X_valid)[:, 1]

# Match the competition's selection fraction
K = max(1, round(len(valid_new) * 500 / 72670))

def evaluate_ranking(scores, name):
    scores = np.asarray(scores)
    labels = y_valid.to_numpy()

    top_indices = np.argsort(-scores, kind="stable")[:K]
    hits = int(labels[top_indices].sum())

    return {
        "model": name,
        "k": K,
        "hits": hits,
        "precision_at_k": hits / K,
        "capture_at_k": hits / max(labels.sum(), 1),
        "average_precision": average_precision_score(
            labels, scores
        ),
        "roc_auc": roc_auc_score(labels, scores)
    }

# Exploit-first baseline, EPSS as tie-breaker
baseline_order = np.lexsort((
    -valid_new["epss"].to_numpy(),
    -valid_new["any_exploit"].to_numpy()
))

baseline_scores = np.empty(len(valid_new))
baseline_scores[baseline_order] = np.arange(
    len(valid_new), 0, -1
)

display(pd.DataFrame([
    evaluate_ranking(baseline_scores, "Exploit-first EPSS"),
    evaluate_ranking(
        valid_new["epss"].to_numpy(), "EPSS"
    ),
    evaluate_ranking(xgb_pred, "Recent-CVE XGBoost")
]).sort_values("hits", ascending=False))


,model,k,hits,precision_at_k,capture_at_k,average_precision,roc_auc
0,Exploit-first EPSS,81,2,0.024691,0.153846,0.011307,0.833703
2,Recent-CVE XGBoost,81,1,0.012346,0.076923,0.011158,0.741605
1,EPSS,81,0,0.000000,0.000000,0.003926,0.700481


In [29]:

import numpy as np
import pandas as pd
from scipy.stats import rankdata

v = valid_new.reset_index(drop=True).copy()

epss = v["epss"].to_numpy()
exploit = v["any_exploit"].to_numpy()
ml_score = np.asarray(xgb_pred)

def rank01(x):
    r = rankdata(x, method="average")
    return (r - 1) / max(len(r) - 1, 1)

epss_rank = rank01(epss)
ml_rank = rank01(ml_score)

results = []
predictions = {}

for weight in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:

    # Combine scores only within exploit groups
    blended = (
        (1 - weight) * epss_rank
        + weight * ml_rank
    )

    # Public exploit status always takes precedence
    ordering = np.lexsort((
        -blended,
        -exploit
    ))

    scores = np.empty(len(v))
    scores[ordering] = np.arange(len(v), 0, -1)

    name = f"Exploit-first + ML {weight:.0%}"

    predictions[name] = scores
    results.append(evaluate_ranking(scores, name))

display(
    pd.DataFrame(results).sort_values(
        ["hits", "average_precision"],
        ascending=False
    )
)


,model,k,hits,precision_at_k,capture_at_k,average_precision,roc_auc
4,Exploit-first + ML 75%,81,2,0.024691,0.153846,0.012874,0.810993
3,Exploit-first + ML 50%,81,2,0.024691,0.153846,0.012540,0.820726
2,Exploit-first + ML 25%,81,2,0.024691,0.153846,0.011947,0.837303
1,Exploit-first + ML 10%,81,2,0.024691,0.153846,0.011622,0.835565
0,Exploit-first + ML 0%,81,2,0.024691,0.153846,0.011307,0.833703
5,Exploit-first + ML 100%,81,1,0.012346,0.076923,0.014267,0.796877


In [30]:

import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, roc_auc_score

DATA = Path("data") if Path("data").exists() else Path("../data")
SEED = 42
ML_WEIGHT = 0.25
HORIZON_DAYS = 90
LABEL_CUTOFF = pd.Timestamp("2025-12-31")

# -----------------------------------------------------
# 1. Ranking and evaluation helpers
# -----------------------------------------------------

def percentile(x):
    x = np.asarray(x, dtype=float)
    return (rankdata(x, method="average") - 1) / max(len(x) - 1, 1)

def make_ranking(frame, ml_predictions=None, weight=0):
    epss = frame["epss"].fillna(0).to_numpy()
    exploit = frame["any_exploit"].to_numpy()

    epss_rank = percentile(epss)

    if ml_predictions is None or weight == 0:
        blended = epss_rank
    else:
        ml_rank = percentile(ml_predictions)
        blended = (1 - weight) * epss_rank + weight * ml_rank

    # Exploit availability always takes precedence.
    # Within each group, sort by the blended score.
    order = np.lexsort((-blended, -exploit))

    scores = np.empty(len(frame), dtype=float)
    scores[order] = np.arange(len(frame), 0, -1)

    return scores

def evaluate(frame, scores, name, fold):
    y = frame["label"].to_numpy()
    k = max(1, round(len(frame) * 500 / 72670))
    top = np.argsort(-scores, kind="stable")[:k]
    hits = int(y[top].sum())

    return {
        "fold": fold,
        "model": name,
        "n": len(frame),
        "positives": int(y.sum()),
        "k": k,
        "hits": hits,
        "precision_at_k": hits / k,
        "capture_at_k": hits / max(y.sum(), 1),
        "average_precision": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores) if len(np.unique(y)) == 2 else np.nan
    }

def build_model(training):
    y = training["label"]
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos

    model = XGBClassifier(
        n_estimators=250,
        max_depth=2,
        learning_rate=0.025,
        min_child_weight=15,
        subsample=0.85,
        colsample_bytree=0.9,
        reg_lambda=30,
        reg_alpha=5,
        scale_pos_weight=min(n_neg / max(n_pos, 1), 100),
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(training[FEATURES], y)
    return model

# -----------------------------------------------------
# 2. Chronological rolling validation
# -----------------------------------------------------

folds = [
    ("2025-Q1", "2025-01", "2025-03"),
    ("2025-Q2", "2025-04", "2025-06"),
    ("2025-Q3", "2025-07", "2025-09"),
]

records = []

for name, start, end in folds:
    valid_fold = cohort[
        cohort["snapshot_month"].between(start, end)
    ].copy().reset_index(drop=True)

    # Restrict training to snapshots with fully mature
    # 90-day labels before this validation period.
    validation_start = pd.Timestamp(start + "-01")
    training_cutoff = validation_start - pd.Timedelta(days=HORIZON_DAYS)

    train_fold = cohort[
        cohort["snapshot_date"] <= training_cutoff
    ].copy()

    # Guard against using labels not yet observable
    train_fold = train_fold[
        train_fold["snapshot_date"]
        + pd.Timedelta(days=HORIZON_DAYS)
        <= validation_start
    ]

    assert train_fold["label"].sum() > 0

    model = build_model(train_fold)
    ml_pred = model.predict_proba(valid_fold[FEATURES])[:, 1]

    baseline = make_ranking(valid_fold)
    reranked = make_ranking(
        valid_fold,
        ml_predictions=ml_pred,
        weight=ML_WEIGHT
    )

    records.extend([
        evaluate(valid_fold, baseline, "Exploit-first EPSS", name),
        evaluate(valid_fold, reranked, "25% ML reranker", name)
    ])

    print(
        f"{name}: train={len(train_fold):,}, "
        f"train positives={train_fold.label.sum()}, "
        f"validation positives={valid_fold.label.sum()}"
    )

fold_results = pd.DataFrame(records)

print("\nPER-FOLD RESULTS")
display(fold_results)

# -----------------------------------------------------
# 3. Aggregate comparison
# -----------------------------------------------------

summary = (
    fold_results.groupby("model")
    .agg(
        total_k=("k", "sum"),
        total_hits=("hits", "sum"),
        mean_ap=("average_precision", "mean"),
        mean_auc=("roc_auc", "mean")
    )
)

summary["pooled_precision_at_k"] = (
    summary["total_hits"] / summary["total_k"]
)

display(summary)

hit_table = fold_results.pivot(
    index="fold",
    columns="model",
    values="hits"
)

display(hit_table)

baseline_hits = summary.loc["Exploit-first EPSS", "total_hits"]
ml_hits = summary.loc["25% ML reranker", "total_hits"]

# Conservative selection rule, fixed in advance:
# Reranker must gain hits overall and lose none in any fold.
no_fold_regression = (
    hit_table["25% ML reranker"]
    >= hit_table["Exploit-first EPSS"]
).all()

use_ml = bool(
    ml_hits > baseline_hits and no_fold_regression
)

print("\nFINAL SELECTION:")
print("25% ML reranker" if use_ml else "Exploit-first EPSS")

# -----------------------------------------------------
# 4. Refit on all eligible historical cohorts
# -----------------------------------------------------

# Ensure 90-day labels have fully matured.
final_train = cohort[
    cohort["snapshot_date"] + pd.Timedelta(days=HORIZON_DAYS)
    <= LABEL_CUTOFF
].copy()

final_model = build_model(final_train)

print(
    f"Final training: {len(final_train):,} rows, "
    f"{final_train.label.sum()} positives"
)

# -----------------------------------------------------
# 5. Prepare the official 2026 scoreboard
# -----------------------------------------------------

scoreboard_final = pd.read_csv(DATA / "scoreboard_2026.csv")

assert scoreboard_final["cve_id"].is_unique
assert len(scoreboard_final) == 72670

scoreboard_final["published"] = pd.to_datetime(
    scoreboard_final["published"]
)

scoreboard_final["snapshot_date"] = (
    pd.to_datetime(scoreboard_final["snapshot_month"] + "-01")
    + pd.offsets.MonthEnd(0)
)

scoreboard_final["age_days"] = (
    scoreboard_final["snapshot_date"]
    - scoreboard_final["published"]
).dt.days

scoreboard_final["epss_missing"] = (
    scoreboard_final["epss"].isna().astype(int)
)

scoreboard_final["epss"] = (
    scoreboard_final["epss"].fillna(0)
)

flags = ["exploitdb", "metasploit", "nuclei", "poc_github"]

scoreboard_final[flags] = (
    scoreboard_final[flags].fillna(0)
)

scoreboard_final["n_exploit_sources"] = (
    scoreboard_final[flags].sum(axis=1)
)

scoreboard_final["any_exploit"] = (
    scoreboard_final["n_exploit_sources"] > 0
).astype(int)

scoreboard_final["log_epss"] = (
    np.log1p(scoreboard_final["epss"] * 1000)
)

scoreboard_final["epss_x_exploit"] = (
    scoreboard_final["epss"]
    * scoreboard_final["any_exploit"]
)

assert scoreboard_final[FEATURES].notna().all().all()

# -----------------------------------------------------
# 6. Generate final top-500 ranking
# -----------------------------------------------------

if use_ml:
    final_pred = final_model.predict_proba(
        scoreboard_final[FEATURES]
    )[:, 1]

    final_scores = make_ranking(
        scoreboard_final,
        ml_predictions=final_pred,
        weight=ML_WEIGHT
    )
else:
    final_scores = make_ranking(scoreboard_final)

scoreboard_final["ranking_score"] = final_scores

top500 = (
    scoreboard_final
    .sort_values("ranking_score", ascending=False)
    [["cve_id"]]
    .head(500)
)

assert len(top500) == 500
assert top500["cve_id"].is_unique

top500.to_csv("top500.csv", index=False)

print("\nSaved top500.csv")
display(top500.head(20))

# Detailed ranking for the demo/dashboard
scoreboard_final.sort_values(
    "ranking_score", ascending=False
).to_csv("full_ranking_2026.csv", index=False)


2025-Q1: train=28,839, train positives=20, validation positives=11
2025-Q2: train=39,868, train positives=32, validation positives=14
2025-Q3: train=48,117, train positives=43, validation positives=13

PER-FOLD RESULTS


,fold,model,n,positives,k,hits,precision_at_k,capture_at_k,average_precision,roc_auc
0,2025-Q1,Exploit-first EPSS,8249,11,57,3,0.052632,0.272727,0.111329,0.719206
1,2025-Q1,25% ML reranker,8249,11,57,3,0.052632,0.272727,0.116601,0.724227
2,2025-Q2,Exploit-first EPSS,11666,14,80,6,0.075000,0.428571,0.135093,0.785420
3,2025-Q2,25% ML reranker,11666,14,80,6,0.075000,0.428571,0.134896,0.809579
4,2025-Q3,Exploit-first EPSS,11702,13,81,2,0.024691,0.153846,0.011307,0.833703
5,2025-Q3,25% ML reranker,11702,13,81,2,0.024691,0.153846,0.012384,0.839698


,total_k,total_hits,mean_ap,mean_auc,pooled_precision_at_k
model,,,,,
25% ML reranker,218,11,0.08796,0.791168,0.050459
Exploit-first EPSS,218,11,0.08591,0.779443,0.050459


model,25% ML reranker,Exploit-first EPSS
fold,,
2025-Q1,3,3
2025-Q2,6,6
2025-Q3,2,2



FINAL SELECTION:
Exploit-first EPSS
Final training: 71,485 rows, 70 positives

Saved top500.csv


,cve_id
62040,CVE-2026-8037
1596,CVE-2022-25369
2072,CVE-2024-45257
40258,CVE-2026-49160
2907,CVE-2025-14558
9550,CVE-2026-10523
2104,CVE-2024-51092
24158,CVE-2026-27771
25345,CVE-2026-29058
20349,CVE-2026-2329
